In [ ]:
# ============================================================
# RF FEATURE IMPORTANCE: GLOBAL + CLASS-SPECIFIC
# ============================================================
# This section computes:
# 1. Global RF feature importance from the trained Random Forest model.
# 2. Class-specific permutation importance using F1-score for each class.
# 3. A combined figure and CSV tables for manuscript reporting.
#
# Required objects from previous RF workflow:
# - rf
# - X_test
# - y_test
# - label_encoder
# - X_COLS
# - FIGURES_DIR
# - TABLES_DIR
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score, make_scorer


# ============================================================
# Define display names
# ============================================================

pretty_feature_map = {
    "PERM_permeability": "Reservoir permeability",
    "PERM_perm_factor_of_failed_well": "Adjacent-well failure factor",
    "PERM_dist_to_well_in_grid": "Injector-to-well distance",
    "PERM_caprock_perm": "Caprock permeability",
    "porosity": "Reservoir porosity",
    "aquifer_porosity": "Aquifer porosity",
    "aquifer_permeability": "Aquifer permeability",
}

pretty_features = [pretty_feature_map.get(col, col) for col in X_COLS]

class_display_map = {
    "low_risk": "Low leakage",
    "high_risk": "High leakage",
    "extreme": "Extreme leakage"
}


# ============================================================
# Define class-specific F1 scorer
# ============================================================

def make_class_f1_scorer(class_label):
    """
    Create a class-specific F1 scorer for permutation importance.

    Parameters
    ----------
    class_label : int
        Encoded class label from LabelEncoder.

    Returns
    -------
    scorer
        Scikit-learn scorer for class-specific F1.
    """
    return make_scorer(
        f1_score,
        labels=[class_label],
        average="macro",
        zero_division=0
    )


# ============================================================
# Get encoded class labels
# ============================================================

class_mapping = dict(
    zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))
)

print("\n=== CLASS MAPPING FOR FEATURE IMPORTANCE ===")
print(class_mapping)

required_classes = ["low_risk", "high_risk", "extreme"]

missing_classes = [cls for cls in required_classes if cls not in class_mapping]
if missing_classes:
    raise ValueError(
        f"Missing required classes in label encoder: {missing_classes}"
    )

low_label = class_mapping["low_risk"]
high_label = class_mapping["high_risk"]
extreme_label = class_mapping["extreme"]


# ============================================================
# Global RF feature importance
# ============================================================

# Global RF feature importance from impurity-based importance.
# These values are always non-negative.
global_imp = np.clip(rf.feature_importances_, 0, None)


# ============================================================
# Class-specific permutation importance
# ============================================================

print("\nComputing class-specific permutation importance...")

perm_low = permutation_importance(
    rf,
    X_test,
    y_test,
    scoring=make_class_f1_scorer(low_label),
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

perm_high = permutation_importance(
    rf,
    X_test,
    y_test,
    scoring=make_class_f1_scorer(high_label),
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

perm_extreme = permutation_importance(
    rf,
    X_test,
    y_test,
    scoring=make_class_f1_scorer(extreme_label),
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

# Raw permutation importance values.
imp_low_raw = perm_low.importances_mean
imp_high_raw = perm_high.importances_mean
imp_extreme_raw = perm_extreme.importances_mean

# Clipped values for plotting/reporting.
# Negative permutation importance values indicate no positive contribution
# under repeated permutations and are set to zero for visualization/reporting.
imp_low = np.clip(imp_low_raw, 0, None)
imp_high = np.clip(imp_high_raw, 0, None)
imp_extreme = np.clip(imp_extreme_raw, 0, None)

print("Class-specific permutation importance calculated.")
print("Raw values are retained in a separate CSV file.")
print("Clipped values are used for manuscript tables and figures.")


# ============================================================
# Sort features for plotting
# ============================================================

mean_imp = (global_imp + imp_low + imp_high + imp_extreme) / 4
sorted_idx = np.argsort(mean_imp)

features_sorted = [pretty_features[i] for i in sorted_idx]

global_sorted = global_imp[sorted_idx]
low_sorted = imp_low[sorted_idx]
high_sorted = imp_high[sorted_idx]
extreme_sorted = imp_extreme[sorted_idx]
mean_sorted = mean_imp[sorted_idx]

low_raw_sorted = imp_low_raw[sorted_idx]
high_raw_sorted = imp_high_raw[sorted_idx]
extreme_raw_sorted = imp_extreme_raw[sorted_idx]


# ============================================================
# Create summary tables
# ============================================================

importance_table_clipped = pd.DataFrame({
    "Feature": features_sorted,
    "Global": global_sorted,
    "Low leakage": low_sorted,
    "High leakage": high_sorted,
    "Extreme leakage": extreme_sorted,
    "Mean_for_sorting": mean_sorted
})

importance_table_raw = pd.DataFrame({
    "Feature": features_sorted,
    "Global": global_sorted,
    "Low leakage raw": low_raw_sorted,
    "High leakage raw": high_raw_sorted,
    "Extreme leakage raw": extreme_raw_sorted
})

# Manuscript table excludes the internal sorting column.
importance_table_manuscript = importance_table_clipped.drop(
    columns=["Mean_for_sorting"]
).copy()

importance_table_display = importance_table_manuscript.copy()
numeric_cols = ["Global", "Low leakage", "High leakage", "Extreme leakage"]
importance_table_display[numeric_cols] = (
    importance_table_display[numeric_cols].round(4)
)

importance_table_clipped.to_csv(
    TABLES_DIR / "Table_6_RF_FeatureImportance_clipped_exact_with_sorting.csv",
    index=False
)

importance_table_manuscript.to_csv(
    TABLES_DIR / "Table_6_RF_FeatureImportance_clipped_exact.csv",
    index=False
)

importance_table_display.to_csv(
    TABLES_DIR / "Table_6_RF_FeatureImportance_clipped_rounded.csv",
    index=False
)

importance_table_raw.to_csv(
    TABLES_DIR / "Table_6_RF_FeatureImportance_raw_values.csv",
    index=False
)

print("\n=== RF FEATURE IMPORTANCE TABLE — ROUNDED VALUES ===")
print(importance_table_display.to_string(index=False))


# ============================================================
# Plot global and class-specific feature importance
# ============================================================

y = np.arange(len(features_sorted))
bar_h = 0.18

fig, ax = plt.subplots(figsize=(12, 7), dpi=600)

ax.barh(
    y - 1.5 * bar_h,
    global_sorted,
    height=bar_h,
    color="#4A90E2",
    label="Global"
)

ax.barh(
    y - 0.5 * bar_h,
    low_sorted,
    height=bar_h,
    color="#FDD835",
    label="Low leakage"
)

ax.barh(
    y + 0.5 * bar_h,
    high_sorted,
    height=bar_h,
    color="#FB8C00",
    label="High leakage"
)

ax.barh(
    y + 1.5 * bar_h,
    extreme_sorted,
    height=bar_h,
    color="#E41A1C",
    label="Extreme leakage"
)

ax.set_yticks(y)
ax.set_yticklabels(features_sorted, fontsize=16)

ax.set_xlabel("Importance score", fontsize=16)
ax.set_ylabel("Input variables", fontsize=16)
ax.set_title("Global and Class-Specific Feature Importance", fontsize=18)

ax.tick_params(axis="x", labelsize=14)
ax.legend(fontsize=13, frameon=False)
ax.grid(axis="x", alpha=0.25, linestyle="--")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

fig_path = FIGURES_DIR / "Figure_6_RF_FeatureImportance_Global_and_ClassSpecific.png"
fig.savefig(fig_path, dpi=600, bbox_inches="tight")

plt.show()
plt.close(fig)

print(f"\nSaved feature importance figure: {fig_path}")


# ============================================================
# Print final summary
# ============================================================

top_global_idx = np.argmax(global_imp)
top_global_feature = pretty_features[top_global_idx]
top_global_value = global_imp[top_global_idx]

print("\n=== FEATURE IMPORTANCE SUMMARY ===")
print(f"Top global feature: {top_global_feature} ({top_global_value:.4f})")
print(f"Saved rounded table: {TABLES_DIR / 'Table_6_RF_FeatureImportance_clipped_rounded.csv'}")
print(f"Saved raw values: {TABLES_DIR / 'Table_6_RF_FeatureImportance_raw_values.csv'}")
print("RF feature importance workflow completed successfully.")